# Writing to Apache Iceberg

Iceberg is another transactional table format. Unlike Delta, Iceberg is **catalog-first** — you don't write to a path, you write to a *table* in a catalog. The catalog tracks which files belong to the current snapshot.

We'll use the simplest catalog: a **Hadoop catalog** rooted at a local folder. No Hive metastore, no extra services.

**Compatibility note**: Spark **4.1.3 / Scala 2.13** → `iceberg-spark-runtime-4.1_2.13:1.12.0`. Match your Spark major.minor version in the artifact name.

First run downloads the jar via `spark.jars.packages` — be patient on the first cell.

## Wipe stale warehouse first

If a previous run left a partial `<table>/` folder under the warehouse, the Hadoop catalog fails with:

```
FileNotFoundException: File .../metadata/version-hint.text does not exist
```

`version-hint.text` is the Hadoop catalog's pointer to the current metadata file — if the directory exists but the metadata is missing, the catalog won't recover on its own. Wiping the warehouse is the cleanest reset for a tutorial notebook. Skip this cell if you want to keep prior tables.

In [1]:
! rm -rf /tmp/iceberg_warehouse

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
warehouse = "file:///tmp/iceberg_warehouse"

spark = (
    SparkSession.builder
    .appName("Write Iceberg")
    .master("local[2]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Define a catalog called 'local'
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", warehouse)
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a5168f73-829a-4154-a88e-de435360b438;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 96ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submi

In [4]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()
df.limit(5).toPandas()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)



,CustomerID,Gender,Age,AnnualIncome,SpendingScore
0,1,Male,19,15000,39
1,2,Male,21,15000,81
2,3,Female,20,16000,6
3,4,Female,23,16000,77
4,5,Female,31,17000,40


## 1. Create namespace (database) and write the table

Three-part name: `<catalog>.<namespace>.<table>` → `local.db.mall_customers`.

In [5]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

DataFrame[]

In [6]:
# DataFrameWriterV2 is the recommended API for Iceberg
df.writeTo("local.db.mall_customers").using("iceberg").createOrReplace()

26/04/25 02:33:20 WARN HadoopTableOperations: Error reading version hint file file:/tmp/iceberg_warehouse/db/mall_customers/metadata/version-hint.text
java.io.FileNotFoundException: File file:/tmp/iceberg_warehouse/db/mall_customers/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:320)
	at org.apache.iceb

In [8]:
# Iceberg lays it out as warehouse/<namespace>/<table>/{data,metadata}
! ls -lh /tmp/iceberg_warehouse/db/mall_customers
! ls -lh /tmp/iceberg_warehouse/db/mall_customers/metadata

total 8.0K
drwxr-xr-x 2 root root 4.0K Apr 25 02:33 data
drwxr-xr-x 2 root root 4.0K Apr 25 02:33 metadata
total 24K
-rw-r--r-- 1 root root 6.8K Apr 25 02:33 6a0a72c5-7f58-4bb6-b6a3-88beae072c3c-m0.avro
-rw-r--r-- 1 root root 4.4K Apr 25 02:33 snap-6600595306130442961-1-6a0a72c5-7f58-4bb6-b6a3-88beae072c3c.avro
-rw-r--r-- 1 root root 2.5K Apr 25 02:33 v1.metadata.json
-rw-r--r-- 1 root root    1 Apr 25 02:33 version-hint.text


## 2. Read it back

In [9]:
spark.table("local.db.mall_customers").count()

200

In [10]:
spark.sql("SELECT * FROM local.db.mall_customers LIMIT 5").show()

+----------+------+---+------------+-------------+
|CustomerID|Gender|Age|AnnualIncome|SpendingScore|
+----------+------+---+------------+-------------+
|         1|  Male| 19|       15000|           39|
|         2|  Male| 21|       15000|           81|
|         3|Female| 20|       16000|            6|
|         4|Female| 23|       16000|           77|
|         5|Female| 31|       17000|           40|
+----------+------+---+------------+-------------+



## 3. Partitioned table

In [11]:
(
    df.writeTo("local.db.mall_customers_by_gender")
      .using("iceberg")
      .partitionedBy(F.col("Gender"))
      .createOrReplace()
)

26/04/25 02:37:32 WARN HadoopTableOperations: Error reading version hint file file:/tmp/iceberg_warehouse/db/mall_customers_by_gender/metadata/version-hint.text
java.io.FileNotFoundException: File file:/tmp/iceberg_warehouse/db/mall_customers_by_gender/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:320)

In [12]:
! ls /tmp/iceberg_warehouse/db/mall_customers_by_gender/data

'Gender=Female'  'Gender=Male'


## 4. Append

In [13]:
df_more = df.limit(5)
df_more.writeTo("local.db.mall_customers").append()

spark.table("local.db.mall_customers").count()

205

## 5. Snapshots and time travel

Each commit creates a new snapshot. Iceberg exposes them as a metadata table `<table>.snapshots`.

In [14]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM local.db.mall_customers.snapshots").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6600595306130442961|2026-04-25 02:33:20.198|append   |
|5424466218773910950|2026-04-25 02:37:36.247|append   |
+-------------------+-----------------------+---------+



In [15]:
# Read a previous snapshot — grab one from the cell above
first_snapshot = spark.sql(
    "SELECT snapshot_id FROM local.db.mall_customers.snapshots ORDER BY committed_at LIMIT 1"
).collect()[0][0]

spark.read \
    .option("versionAsOf", first_snapshot) \
    .table("local.db.mall_customers") \
    .count()

200

## 6. Update / Delete via SQL

In [16]:
spark.sql("""
    UPDATE local.db.mall_customers
    SET SpendingScore = SpendingScore + 1
    WHERE Gender = 'Female'
""")

DataFrame[]

In [17]:
spark.sql("""
    DELETE FROM local.db.mall_customers
    WHERE Age > 65
""")

DataFrame[]

## 7. Inspect history and files

In [18]:
spark.sql("SELECT * FROM local.db.mall_customers.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-25 02:33:20.198|6600595306130442961|NULL               |true               |
|2026-04-25 02:37:36.247|5424466218773910950|6600595306130442961|true               |
|2026-04-25 02:37:40.624|302856566543830737 |5424466218773910950|true               |
|2026-04-25 02:37:41.431|219706817217330366 |302856566543830737 |true               |
+-----------------------+-------------------+-------------------+-------------------+



In [19]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM local.db.mall_customers.files").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                       |record_count|file_size_in_bytes|
+----------------------------------------------------------------------------------------------------------------+------------+------------------+
|file:/tmp/iceberg_warehouse/db/mall_customers/data/00000-24-6b03d045-cb8a-46eb-bc6e-084f759debec-0-00001.parquet|193         |2846              |
+----------------------------------------------------------------------------------------------------------------+------------+------------------+



In [20]:
spark.stop()